# BES Playbook: Model Robustness Evaluation using Adversarial Robustness Toolbox (ART)

### Overview
This notebook demonstrates how to evaluate **model robustness** against **adversarial attacks** 
using the **Adversarial Robustness Toolbox (ART)** — integrated within a managed **BES environment**.

It leverages the modular codebase developed under:
  $BESMAN_ENV_DIR/tools/adversarial-robustness-toolbox/examples/be-secure/demo/



## 1. Environment Information

This playbook assumes that a **BES-managed machine learning environment** has already been installed using the **`bes-env`** setup process:

| Variable | Description | Example |
|-----------|--------------|----------|
| `BESMAN_ENV_DIR` | Root environment directory | `$HOME/ml-assessment-env` |
| `BESMAN_VENV_NAME` | Virtual environment name | `ml-assessment-venv` |
| `BESMAN_TOOLS_DIR` | Directory containing cloned tools | `$BESMAN_ENV_DIR/tools` |
| `BESMAN_ART_REPO` | ART GitHub repository | `https://github.com/NeerajK007/adversarial-robustness-toolbox.git` |

**ART and its dependencies** are already installed inside the virtual environment:
```bash
source $HOME/ml-assessment-env/ml-assessment-venv/bin/activate
pip show adversarial-robustness-toolbox



## 2. Dependencies and Supporting Modules

The robustness evaluation pipeline depends on several helper modules and datasets:

File	Purpose
model_registry.py	Central registry of model definitions and metadata.
model_loader.py	Loads model from registry, caches them, and prepares for inference.
demo_attack.py	Wrapper around ART attack implementations (FGSM, PGD, C&W).
datasets/mvtec_loader.py	MVTec dataset loader for tile defect classification.
datasets/traffic_sign_loader.py	Dataset loader for Indian traffic sign classification.
evaluate_robustness.py	Main driver that loads models, executes attacks, and generates robustness report.


In [15]:
%%bash
## 3.The whole cell is now executed by the shell
source $HOME/ml-assessment-env/ml-assessment-venv/bin/activate
python3 --version
which python3

Python 3.12.7
/home/neeraj/ml-assessment-env/ml-assessment-venv/bin/python3


## Step 4 — Configuring Models, Datasets, and Attacks

The script performs:
Model Loading — via model_loader.load_demo_model().
Dataset Loading — via TrafficSignDataset or MVTecSyntheticDataset.
Baseline Evaluation — computes clean accuracy & confidence.
Attack Execution — runs FGSM, PGD, and C&W attacks sequentially.
Report Generation — produces a standardized JSON report in console output.

Before running robustness evaluation, you can control **which model**, **dataset**, and **adversarial attacks**
are tested through configuration blocks inside `evaluate_robustness.py` and `model_registry.py`.

---

### 🧠 Model & Dataset Configuration
All available demo models and datasets are defined in `model_registry.py`.

Each model entry includes:
| Key | Description |
|-----|--------------|
| `model_fn` | Base architecture (e.g., `mobilenet_v2`) |
| `weights_path` | Path to fine-tuned model weights |
| `adv_weights_path` | Path to adversarially-trained weights (if any) |
| `num_classes` | Output classes for the dataset |
| `dataset_name` | Dataset used for evaluation |
| `variant_supported` | Available variants (`normal`, `adv_trained`, etc.) |
| `demo_type` | Identifier string used throughout the project |

**Example:**
```python
"Indian-trafic-signal-misclassification": {
    "model_fn": models.mobilenet_v2,
    "num_classes": 4,
    "weights_path": "weights/20250919_133106_mobilenetv2_traffic_signs.pth",
    "adv_weights_path": "weights/20250918_154508_mobilenetv2_traffic_signs_AdvTrained.pth",
    "dataset_name": "Indian_traffic_sign_classification_dataset",
    "variant_supported": ["normal", "adv_trained"],
    "demo_type": "Indian-trafic-signal-misclassification"
}


# To switch models, simply change:

"demo_type": "Tile-defect-misclassification"

in your get_config() function (within evaluate_robustness.py).


In [16]:
## 5. Run Model Robustness Evaluation

import os
import subprocess
from datetime import datetime

# --- Setup Paths ---
demo_dir = os.path.expanduser("~/ml-assessment-env/tools/adversarial-robustness-toolbox/examples/be-secure/demo")
eval_script = os.path.join(demo_dir, "evaluate_robustness.py")

# --- Confirm working directory ---
os.chdir(demo_dir)
print(f"Working directory set to:\n{os.getcwd()}")

# --- Sanity check ---
if not os.path.exists(eval_script):
    raise FileNotFoundError(f"Could not find evaluate_robustness.py at: {eval_script}")

# --- Run script in subprocess ---
print(f"\nRunning model robustness evaluation at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} ...\n")

result = subprocess.run(
    ["python3", eval_script],
    capture_output=True,
    text=True
)

# --- Display outputs neatly ---
if result.returncode == 0:
    print("Evaluation completed successfully!\n")
else:
    print("Evaluation script exited with errors.\n")

# Print STDOUT logs (main output)
print("---- STDOUT ----")
print(result.stdout)

# Print STDERR logs (error logs, if any)
if result.stderr.strip():
    print("\n---- STDERR ----")
    print(result.stderr)


Working directory set to:
/home/neeraj/ml-assessment-env/tools/adversarial-robustness-toolbox/examples/be-secure/demo

Running model robustness evaluation at 2025-10-13 14:45:52 ...

Evaluation completed successfully!

---- STDOUT ----
2025-10-13 14:45:57,358 [INFO] Classes: {'SCHOOL_AHEAD': 0, 'SPEED_LIMIT_70': 1, 'SPEED_LIMIT_80': 2, 'STOP': 3}
2025-10-13 14:45:57,359 [INFO] Loaded 149 samples for test


========== FINAL FORMATTED REPORT ==========
{
  "REPORT_META": {
    "report_id": "20251013_144557",
    "report_type": "adversarial_robustness",
    "generated_at": "2025-10-13T14:50:09.468489",
    "dataset": {
      "name": "Indian_traffic_sign_classification_dataset",
      "size_test": 149,
      "num_classes": "unknown",
      "source": "datasets url"
    },
    "model": {
      "pretrained_on": "ImageNet",
      "fine_tuned_on": "Indian_traffic_sign_classification_dataset",
      "variant": "normal",
      "weights_file": "weights/20250919_133106_mobilenetv2_traffic_signs.pth

In [17]:
import os, json, re

# --- User can set report path here ---
report_path = "/home/neeraj/besecure-ml-assessment-datastore/models/mobilenet_v2/fuzz-test/evasion/VulnerabilityReport.json"

# Ensure directory exists
os.makedirs(os.path.dirname(report_path), exist_ok=True)

# Extract the JSON report from previous stdout
stdout_text = result.stdout
match = re.search(
    r"========== FINAL FORMATTED REPORT ==========\n(.*?)\n============================================",
    stdout_text, re.S
)

if match:
    try:
        report_data = json.loads(match.group(1).strip())
        with open(report_path, "w") as f:
            json.dump(report_data, f, indent=2)
        print(f"Report saved: {report_path} ({os.path.getsize(report_path)} bytes)")
    except json.JSONDecodeError as e:
        print("Invalid JSON format in output:", e)
else:
    print("No JSON report found in evaluation output.")


Report saved: /home/neeraj/besecure-ml-assessment-datastore/models/mobilenet_v2/fuzz-test/evasion/VulnerabilityReport.json (2945 bytes)
